# Gans Pipeline

## Libraries and Constants

In [18]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

In [19]:
load_dotenv()
connection_string = os.getenv("CON_STRING")
openweather_key = os.getenv("OPENWEATHER_KEY")
rapidapi_key = os.getenv("RAPIDAPI_KEY")

wiki_headers = {'User-Agent': 'Chrome/134.0.0.0'}
rapidapi_headers = {"x-rapidapi-key": rapidapi_key, "x-rapidapi-host": "aerodatabox.p.rapidapi.com","Content-Type": "application/json"}

## Define Functions

### Cities

In [20]:
def get_cities_info(cities):
    countries = []
    latitudes = []
    longitudes = []

    for city in cities:
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=wiki_headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            for row in city_soup.find("table").find_all("tr"):
                if row.find(string="Country"):
                    countries.append(row.find("a").get_text())
            lat, lon = city_soup.find("table").find(class_="geo").get_text().split("; ")
            try:
                latitudes.append(float(lat))
                longitudes.append(float(lon))
            except ValueError:
                latitudes.append(None)
                longitudes.append(None)

        else:
            print(f"WARNING: Could not retrieve HTML for {city}")

    cities_df = pd.DataFrame({"name": cities, "country": countries, "latitude": latitudes, "longitude": longitudes})
    
    return cities_df

### Populations

In [21]:
def get_populations(cities_df):
    populations = []
    city_ids = []

    for i, row in cities_df.iterrows():
        city = row["name"]
        city_id = row["city_id"]
        
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=wiki_headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            for row in city_soup.find("table").find_all("tr"):
                if row.find(string="Population"):
                    population = int(row.find_next("td").get_text().replace(",", ""))
        populations.append(population)
        city_ids.append(city_id)

    pops_df = pd.DataFrame({"city_id": city_ids, "population": populations, "date_gathered": pd.Timestamp.now().date()})
    
    return pops_df

### Weather Forecasts

In [22]:
def get_forecasts(cities_df):
    forecasts = []
    url = "https://api.openweathermap.org/data/2.5/forecast"
    
    # loop through cities_df
    for _, row in cities_df.iterrows():
        params = {"lat": row["latitude"], "lon" :row["longitude"], "appid":openweather_key, "units": "metric"}
        response = requests.get(url, params=params)
        response_json = response.json()
        
        for forecast in response_json['list']:
            forecast_data = {
                "temp": forecast['main']['temp'],
                "feels_like": forecast['main']['feels_like'],
                "humidity_%": forecast['main']['humidity'],
                "wind_speed": forecast['wind']['speed'],
                "wind_gust": forecast['wind']['gust'],
                "precipitation_%": forecast['pop'],
                "rain_3h": forecast.get('rain', {}).get('3h', 0),
                "snow_3h": forecast.get('snow', {}).get('3h', 0),
                "forecast_time": forecast['dt_txt'],
                "city_id": row["city_id"] # add city_id to reference back to 'cities' table
            }
            forecasts.append(forecast_data)
    
    forecasts_df = pd.DataFrame(forecasts)
    
    return forecasts_df

### Airports

In [23]:
def get_airports(cities_df):
    all_airports = []
    
    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
    for _, row in cities_df.iterrows():
        querystring = {"lat":row["latitude"],"lon":row["longitude"],"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}
        response = requests.get(url, headers=rapidapi_headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            airports["city_id"] = row["city_id"] # add city_id for foreign key reference
            all_airports.append(airports)
        else:
            print(f"WARNING: Failed to retrieve airports for {row["name"]}")

    airports_df = pd.concat(all_airports, ignore_index=True) # make one DataFrame from individual results
    airports_df = airports_df[["icao", "name", "city_id"]] 

    return airports_df

### Flights

In [24]:
def get_flights(airports_df):
    arrivals = [] # empty list to store arrivals

    tomorrow = pd.Timestamp.now().date()+pd.Timedelta(1, "day")
    tomorrow_str = tomorrow.strftime("%Y-%m-%d")
    
    times = [["00:00", "11:59"], ["12:00", "23:59"]]
    for _, row in airports_df.iterrows():
        for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{row["icao"]}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
            querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
            response = requests.get(url, headers=rapidapi_headers, params=querystring)
        
            if response.status_code == 200:
                data = response.json()["arrivals"]
                for arrival in data:
                    arrival_dict = {
                        "arrive_icao": row["icao"], # add foreign-key information
                        "depart_icao": arrival["movement"]["airport"].get("icao", None), # add this since countryCode and maybe name aren't reliable
                        "depart_airport": arrival["movement"]["airport"].get("name", None),
                        "depart_country": arrival["movement"]["airport"].get("countryCode", None), # move .upper() to deal with NoneTypes
                        "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                        "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                        "flight_number": arrival.get("number", None),
                        "aircraft": arrival.get("aircraft", {}).get("model", None)
                    }
                    arrivals.append(arrival_dict)
            else:
                print(f"WARNING: Failed to retrieve airports for {row["name"]}")
    
    flights_df = pd.DataFrame(arrivals)
    
    # clean DataFrame
    flights_df["depart_country"] = flights_df["depart_country"].str.upper()
    flights_df["arrive_time_scheduled"] = pd.to_datetime(flights_df["arrive_time_scheduled"])
    flights_df["arrive_time_revised"] = pd.to_datetime(flights_df["arrive_time_revised"])
    
    return flights_df

## Add New Cities

In [25]:
cities = ["Berlin", "Hamburg", "Munich"]

cities_df = get_cities_info(cities)
cities_df.to_sql(
    "cities",
    con=connection_string,
    if_exists="append",
    index=False
)

3

In [32]:
cities_df = pd.read_sql(
    "SELECT * FROM cities WHERE city_id NOT IN (SELECT DISTINCT city_id FROM populations);", 
    con=connection_string
)

pops_df = get_populations(cities_df)
pops_df.to_sql(
    "populations",
    con=connection_string,
    if_exists="append",
    index=False
)

airports_df = get_airports(cities_df)
airports_df.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

airports_df = pd.DataFrame({
    "icao": ["EDDB", "EDDT"],
    "name": ["Berlin Brandenburg", "Berlin -Tegel"],
    "city_id": [1, 1]
})

airports_df.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

ValueError: No objects to concatenate

## Occasional Updates

In [33]:
cities_df = pd.read_sql("cities", con=connection_string)

pops_df = get_populations(cities_df)
pops_df.to_sql(
    "populations",
    con=connection_string,
    if_exists="append",
    index=False
)

airports_df = get_airports(cities_df)
airports_df.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

ValueError: No objects to concatenate

## Daily Updates

In [34]:
cities_df = pd.read_sql("cities", con=connection_string)
airports_df = pd.read_sql("SELECT * FROM airports WHERE `active` = 1", con=connection_string)

forecasts_df = get_forecasts(cities_df)
forecasts_df.to_sql(
    "forecasts",
    con=connection_string,
    if_exists="append",
    index=False    
)

flights_df = get_flights(airports_df)
flights_df.to_sql(
    "flights",
    con=connection_string,
    if_exists="append",
    index=False    
)

KeyError: 'depart_country'